# TRAINING

## Setup and Imports

In [1]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from tqdm import tqdm
from collections import Counter

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


## Define Relation Labels and Run 1 Configuration

This section defines the legal entity categories, legal relation predicates, and the main training configuration for **Run 1**.

In this run, the model is trained on:
- **Gold** annotations
- **Silver** annotations
- **Silver 2025** annotations

However, these sources are not treated equally. Since the annotation quality is different, each training example will later receive a **sample weight**:
- **Gold** examples receive the highest weight
- **Silver StudentA** examples receive an intermediate weight
- **Silver StudentB** examples receive a lower weight
- **Silver 2025** examples receive a lower-to-intermediate weight

This makes the training setup more robust to annotation noise while still exploiting the larger amount of available supervision.

For this first run:
- the training set uses a moderate number of negative examples
- the development set uses fewer negatives, so evaluation is less dominated by the `"no relation"` class
- the best checkpoint will be selected using a **positive-class F1 metric**

In [2]:
import re

LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}

def norm_ent(label: str) -> str:
    if label is None:
        return ""
    lab = str(label).strip()
    if lab.lower() == "ddf":
        return "DDF"
    return lab

def norm_span(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

RELATION_LABELS = [
    "no relation",
    "administered",
    "affect",
    "change abundance",
    "change effect",
    "change expression",
    "compared to",
    "impact",
    "influence",
    "interact",
    "is a",
    "is linked to",
    "located in",
    "part of",
    "produced by",
    "strike",
    "target",
    "used by"
]

label2id = {label: idx for idx, label in enumerate(RELATION_LABELS)}
id2label = {idx: label for idx, label in enumerate(RELATION_LABELS)}

print(f"Total relation labels: {len(RELATION_LABELS)}")
print(f"Labels: {RELATION_LABELS}")

LEGAL_RELATIONS = [
    ("DDF", "affect", "DDF"),
    ("microbiome", "is linked to", "DDF"),
    ("DDF", "target", "human"),
    ("drug", "change effect", "DDF"),
    ("DDF", "is a", "DDF"),
    ("microbiome", "located in", "human"),
    ("chemical", "influence", "DDF"),
    ("dietary supplement", "influence", "DDF"),
    ("DDF", "target", "animal"),
    ("chemical", "impact", "microbiome"),
    ("anatomical location", "located in", "animal"),
    ("microbiome", "located in", "animal"),
    ("chemical", "located in", "anatomical location"),
    ("bacteria", "part of", "microbiome"),
    ("DDF", "strike", "anatomical location"),
    ("drug", "administered", "animal"),
    ("bacteria", "influence", "DDF"),
    ("drug", "impact", "microbiome"),
    ("DDF", "change abundance", "microbiome"),
    ("microbiome", "located in", "anatomical location"),
    ("microbiome", "used by", "biomedical technique"),
    ("chemical", "produced by", "microbiome"),
    ("dietary supplement", "impact", "microbiome"),
    ("bacteria", "located in", "animal"),
    ("animal", "used by", "biomedical technique"),
    ("chemical", "impact", "bacteria"),
    ("chemical", "located in", "animal"),
    ("food", "impact", "bacteria"),
    ("microbiome", "compared to", "microbiome"),
    ("human", "used by", "biomedical technique"),
    ("bacteria", "change expression", "gene"),
    ("chemical", "located in", "human"),
    ("drug", "interact", "chemical"),
    ("food", "administered", "human"),
    ("DDF", "change abundance", "bacteria"),
    ("chemical", "interact", "chemical"),
    ("chemical", "part of", "chemical"),
    ("dietary supplement", "impact", "bacteria"),
    ("DDF", "interact", "chemical"),
    ("food", "impact", "microbiome"),
    ("food", "influence", "DDF"),
    ("bacteria", "located in", "human"),
    ("dietary supplement", "administered", "human"),
    ("bacteria", "interact", "chemical"),
    ("drug", "change expression", "gene"),
    ("drug", "impact", "bacteria"),
    ("drug", "administered", "human"),
    ("anatomical location", "located in", "human"),
    ("dietary supplement", "change expression", "gene"),
    ("chemical", "change expression", "gene"),
    ("bacteria", "interact", "bacteria"),
    ("drug", "interact", "drug"),
    ("microbiome", "change expression", "gene"),
    ("bacteria", "interact", "drug"),
    ("food", "change expression", "gene")
]

legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    s = norm_ent(s)
    o = norm_ent(o)
    legal_pairs.setdefault((s, o), set()).add(p)

print(f"\nTotal legal relation patterns: {len(LEGAL_RELATIONS)}")
print(f"Total unique entity type pairs: {len(legal_pairs)}")


Total relation labels: 18
Labels: ['no relation', 'administered', 'affect', 'change abundance', 'change effect', 'change expression', 'compared to', 'impact', 'influence', 'interact', 'is a', 'is linked to', 'located in', 'part of', 'produced by', 'strike', 'target', 'used by']

Total legal relation patterns: 55
Total unique entity type pairs: 52


# Run 1 configuration

In [3]:

model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
output_model_dir = "../../models/bert_biomedbert_re_A2_weighted_fixed"

max_length = 512
WINDOW_CHARS = 300

# Run 1: moderate negatives for train, fewer for dev
TRAIN_NEGATIVE_SAMPLE_MULTIPLIER = 5
DEV_NEGATIVE_SAMPLE_MULTIPLIER = 1

MAX_PAIR_CHARS = 400

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")
print(f"Train negative multiplier: {TRAIN_NEGATIVE_SAMPLE_MULTIPLIER}")
print(f"Dev negative multiplier: {DEV_NEGATIVE_SAMPLE_MULTIPLIER}")
print(f"Window chars: {WINDOW_CHARS}")


Model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Output directory: ../../models/bert_biomedbert_re_A2_weighted_fixed
Train negative multiplier: 5
Dev negative multiplier: 1
Window chars: 300


## Load Data and Attach Annotation-Quality Metadata

This step loads the JSON files and attaches metadata describing the source of each document.

For **Run 1**, the training data includes:
- Gold
- Silver
- Silver 2025

Each document is enriched with:
- `source_quality`: the dataset source (`gold`, `silver`, `silver_2025`)
- `annotator_group`: a coarse annotation group when available (`expert`, `student_A`, `student_B`, or `unknown`)

This metadata will later be converted into a **sample weight** for each training example.

In [4]:
def infer_source_quality_from_path(path: str) -> str:
    p = path.lower()
    if "gold" in p:
        return "gold"
    if "silver_2025" in p:
        return "silver_2025"
    if "silver" in p:
        return "silver"
    if "bronze" in p:
        return "bronze"
    return "unknown"


def infer_annotator_group(article: dict, source_quality: str) -> str:
    metadata = article.get("metadata", {})
    raw = str(
        metadata.get("annotator", "")
        or metadata.get("annotator_group", "")
        or metadata.get("annotators", "")
    ).lower()

    if source_quality == "gold":
        return "expert"

    if "studenta" in raw or "student_a" in raw or "student a" in raw:
        return "student_A"
    if "studentb" in raw or "student_b" in raw or "student b" in raw:
        return "student_B"
    if "expert" in raw:
        return "expert"

    return "unknown"


def load_re_data(file_paths):
    """Load relation extraction data from multiple JSON files and attach source metadata."""
    all_data = {}

    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            source_quality = infer_source_quality_from_path(file_path)

            for pmid, article in data.items():
                article = dict(article)
                article["_source_quality"] = source_quality
                article["_annotator_group"] = infer_annotator_group(article, source_quality)
                all_data[pmid] = article

            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)} [{source_quality}]")
        else:
            print(f"Warning: {file_path} not found")

    return all_data

def get_example_weight(source_quality: str, annotator_group: str) -> float:
    if source_quality == "gold":
        return 1.0
    if source_quality == "silver":
        return 0.9
    if source_quality == "silver_2025":
        return 0.8
    if source_quality == "bronze":
        return 0.4
    return 0.9

print("Data loading functions with source metadata defined")

Data loading functions with source metadata defined


## Load Training and Development Data

The training set for **Run 1** combines multiple annotation-quality sources:
- Gold
- Silver
- Silver 2025

The development set is loaded separately and kept unchanged as the evaluation reference.

At this stage, each loaded training document already carries source metadata that will later be used for weighted training.

In [5]:
train_files = [
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/bronze_quality/json_format/train_bronze.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver_2025.json",
]

train_data = load_re_data(train_files)
print(f"\nTotal training documents: {len(train_data)}")

dev_data = load_re_data([
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json"
])
print(f"Total dev documents: {len(dev_data)}")

Loaded 639 documents from train_gold.json [gold]
Loaded 811 documents from train_silver.json [silver]
Loaded 2972 documents from train_bronze.json [bronze]
Loaded 499 documents from train_silver_2025.json [silver_2025]

Total training documents: 4921
Loaded 80 documents from dev.json [unknown]
Total dev documents: 80


## Prepare Relation Extraction Examples with Weighted Supervision

This step converts each document into mention-level relation examples.

For each document:
1. positive relation examples are extracted from the gold/silver annotations
2. negative relation candidates are generated from legal entity pairs that are not annotated as related
3. each example receives a `sample_weight` based on annotation quality

This is useful because the dataset mixes cleaner and noisier supervision sources.
Instead of discarding weaker annotations entirely, **Run 1** keeps them but gives them less influence during optimization.

In addition:
- the training set uses more negatives
- the development set uses fewer negatives, so validation is less dominated by `"no relation"`

In [6]:
from collections import defaultdict

def create_full_text_with_offsets(title, abstract):
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1
    return full_text, abstract_offset


def adjust_entity_positions(entity, abstract_offset):
    if entity['location'] == 'abstract':
        return {
            'start_idx': entity['start_idx'] + abstract_offset,
            'end_idx': entity['end_idx'] + abstract_offset,
            'text_span': entity['text_span'],
            'label': entity['label']
        }
    else:
        return {
            'start_idx': entity['start_idx'],
            'end_idx': entity['end_idx'],
            'text_span': entity['text_span'],
            'label': entity['label']
        }


def char_distance(a, b):
    return abs(a["start_idx"] - b["start_idx"])


def prepare_re_examples(data, negative_multiplier=1, legal_pairs=None):
    """
    Prepare mention-level RE examples with source-aware sample weights.
    """
    def loc_rank(loc: str) -> int:
        return 0 if loc == "title" else 1

    def best_pair(subj_cands, obj_cands):
        best = None
        best_score = None

        for s in subj_cands:
            for o in obj_cands:
                if s["start_idx"] == o["start_idx"] and s["end_idx"] == o["end_idx"] and s["location"] == o["location"]:
                    continue

                loc_combo = loc_rank(s["location"]) + loc_rank(o["location"])
                same_loc = 0 if s["location"] == o["location"] else 1
                dist = abs(s["start_idx"] - o["start_idx"])
                score = (loc_combo, same_loc, dist)

                if best_score is None or score < best_score:
                    best_score = score
                    best = (s, o)

        return best

    examples = []

    for pmid, article in tqdm(data.items(), desc="Preparing RE examples"):
        title = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        entities = article["entities"]
        relations = article.get("mention_level_relations", [])

        source_quality = article.get("_source_quality", "unknown")
        annotator_group = article.get("_annotator_group", "unknown")
        sample_weight = get_example_weight(source_quality, annotator_group)

        adjusted_entities = [
            {
                **adjust_entity_positions(e, abstract_offset),
                "label": norm_ent(e["label"]),
                "text_span": norm_span(e["text_span"]),
                "location": e["location"],
            }
            for e in entities
        ]

        ent_index = defaultdict(list)
        for e in adjusted_entities:
            ent_index[(e["text_span"], e["label"])].append(e)

        positive_pairs = set()

        for relation in relations:
            subj_text = norm_span(relation["subject_text_span"])
            obj_text  = norm_span(relation["object_text_span"])
            subj_lab  = norm_ent(relation["subject_label"])
            obj_lab   = norm_ent(relation["object_label"])
            pred      = relation["predicate"].strip()

            if pred not in LEGAL_RELATION_LABELS:
                continue
            if subj_lab not in LEGAL_ENTITY_LABELS or obj_lab not in LEGAL_ENTITY_LABELS:
                continue

            subj_cands = ent_index.get((subj_text, subj_lab), [])
            obj_cands  = ent_index.get((obj_text, obj_lab), [])

            pair = best_pair(subj_cands, obj_cands)
            if not pair:
                continue

            subject, obj = pair
            type_pair = (subject["label"], obj["label"])

            if legal_pairs is not None and type_pair not in legal_pairs:
                continue

            examples.append({
                "text": full_text,
                "subject": subject,
                "object": obj,
                "predicate": pred,
                "pmid": pmid,
                "sample_weight": sample_weight,
                "source_quality": source_quality,
                "annotator_group": annotator_group,
            })

            pair_key = (subject["start_idx"], subject["end_idx"], obj["start_idx"], obj["end_idx"])
            positive_pairs.add(pair_key)

        num_negatives = len(positive_pairs) * negative_multiplier
        negative_candidates = []

        if num_negatives > 0:
            for i, subj in enumerate(adjusted_entities):
                for j, obj in enumerate(adjusted_entities):
                    if i == j:
                        continue

                    type_pair = (subj["label"], obj["label"])
                    if legal_pairs is not None and type_pair not in legal_pairs:
                        continue
                    if char_distance(subj, obj) > MAX_PAIR_CHARS:
                        continue

                    pair_key = (subj["start_idx"], subj["end_idx"], obj["start_idx"], obj["end_idx"])
                    if pair_key in positive_pairs:
                        continue

                    negative_candidates.append({
                        "text": full_text,
                        "subject": subj,
                        "object": obj,
                        "predicate": "no relation",
                        "pmid": pmid,
                        "sample_weight": sample_weight,
                        "source_quality": source_quality,
                        "annotator_group": annotator_group,
                    })

            if negative_candidates:
                num_to_sample = min(num_negatives, len(negative_candidates))
                examples.extend(random.sample(negative_candidates, num_to_sample))

    return examples

## Build Train and Dev Examples

The training and development examples are generated using the same candidate-building logic, but with different negative sampling settings:

- **Training** uses more negative examples to teach the model to reject plausible but false pairs
- **Development** uses fewer negatives to make validation more informative for the positive relation classes

This setup is better suited to relation extraction than using the same negative ratio everywhere.

In [ ]:
print("Preparing training examples...")
train_examples = prepare_re_examples(
    train_data,
    negative_multiplier=TRAIN_NEGATIVE_SAMPLE_MULTIPLIER,
    legal_pairs=legal_pairs
)

positive_count = sum(1 for ex in train_examples if ex['predicate'] != 'no relation')
negative_count = sum(1 for ex in train_examples if ex['predicate'] == 'no relation')

print(f"\nTraining examples prepared: {len(train_examples)}")
print(f"  Positive examples: {positive_count}")
print(f"  Negative examples: {negative_count}")
print(f"  Ratio (neg/pos): {negative_count / max(positive_count, 1):.2f}")

print("\nPreparing dev examples...")
dev_examples = prepare_re_examples(
    dev_data,
    negative_multiplier=DEV_NEGATIVE_SAMPLE_MULTIPLIER,
    legal_pairs=legal_pairs
)

positive_count_dev = sum(1 for ex in dev_examples if ex['predicate'] != 'no relation')
negative_count_dev = sum(1 for ex in dev_examples if ex['predicate'] == 'no relation')

print(f"\nDev examples prepared: {len(dev_examples)}")
print(f"  Positive examples: {positive_count_dev}")
print(f"  Negative examples: {negative_count_dev}")
print(f"  Ratio (neg/pos): {negative_count_dev / max(positive_count_dev, 1):.2f}")

print("\nTraining source distribution:")
source_counter = Counter(ex["source_quality"] for ex in train_examples)
for k, v in source_counter.items():
    print(f"  {k}: {v}")

print("\nTraining annotator-group distribution:")
ann_counter = Counter(ex["annotator_group"] for ex in train_examples)
for k, v in ann_counter.items():
    print(f"  {k}: {v}")

Preparing training examples...


Preparing RE examples: 100%|██████████| 4921/4921 [00:01<00:00, 2705.31it/s]



Training examples prepared: 319650
  Positive examples: 53791
  Negative examples: 265859
  Ratio (neg/pos): 4.94

Preparing dev examples...


Preparing RE examples: 100%|██████████| 80/80 [00:00<00:00, 4597.38it/s]

## Initialize Tokenizer and Entity Marker Tokens

This step initializes the tokenizer for the pretrained biomedical BERT model and adds the special entity marker tokens used by the relation extraction model.

The following markers are introduced:
- `[E1] ... [/E1]` for the subject mention
- `[E2] ... [/E2]` for the object mention

These tokens allow the model to explicitly identify which two entity mentions should be considered when predicting the relation label.

In [ ]:
print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

special_tokens = {
    "additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]
}
tokenizer.add_special_tokens(special_tokens)

e1_token_id = tokenizer.convert_tokens_to_ids("[E1]")
e2_token_id = tokenizer.convert_tokens_to_ids("[E2]")

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocabulary size: {len(tokenizer)}")
print(f"[E1] token id: {e1_token_id}")
print(f"[E2] token id: {e2_token_id}")

## Insert Entity Markers into the Text

This function inserts special markers around the subject and object mentions.

The markers are added directly into the text before tokenization:
- subject mention → `[E1] ... [/E1]`
- object mention → `[E2] ... [/E2]`

This helps the encoder focus on the exact entity pair associated with the candidate relation.

In [ ]:
def insert_entity_markers(text, subject, obj):
    """
    Insert entity marker tokens around subject and object entities.

    Args:
        text: Full text
        subject: Subject entity dict with start_idx and end_idx
        obj: Object entity dict with start_idx and end_idx

    Returns:
        Text with entity markers inserted.
    """
    entities = [
        (subject["start_idx"], subject["end_idx"], "[E1]", "[/E1]"),
        (obj["start_idx"], obj["end_idx"], "[E2]", "[/E2]")
    ]
    entities = sorted(entities, key=lambda x: x[0])

    marked_text = text
    offset = 0

    for start, end, start_marker, end_marker in entities:
        adj_start = start + offset
        adj_end = end + offset + 1  # inclusive end_idx

        marked_text = (
            marked_text[:adj_start]
            + start_marker
            + marked_text[adj_start:adj_end]
            + end_marker
            + marked_text[adj_end:]
        )

        offset += len(start_marker) + len(end_marker)

    return marked_text

## Build a Local Text Window Around the Entity Pair

Instead of always encoding the full title-plus-abstract text, this function extracts a local character window around the subject and object mentions.

This has two advantages:
- it reduces the risk that entity markers are truncated
- it keeps the encoder focused on the most relevant local context

The entity offsets are recomputed so they remain valid inside the extracted window.

In [ ]:
def build_window_around_entities(text, subject, obj, window_chars=300):
    """
    Build a substring window around subject and object mentions.
    Recomputes entity offsets within the extracted window.

    Args:
        text: Full text
        subject: Subject entity dict
        obj: Object entity dict
        window_chars: Number of context characters to keep on each side

    Returns:
        window_text, shifted_subject, shifted_object
    """
    s_start, s_end = subject["start_idx"], subject["end_idx"]
    o_start, o_end = obj["start_idx"], obj["end_idx"]

    left = min(s_start, o_start)
    right = max(s_end, o_end)

    win_start = max(0, left - window_chars)
    win_end = min(len(text) - 1, right + window_chars)

    window_text = text[win_start:win_end + 1]

    subj_w = dict(subject)
    obj_w = dict(obj)

    subj_w["start_idx"] = s_start - win_start
    subj_w["end_idx"] = s_end - win_start
    obj_w["start_idx"] = o_start - win_start
    obj_w["end_idx"] = o_end - win_start

    for ent in (subj_w, obj_w):
        ent["start_idx"] = max(0, min(ent["start_idx"], len(window_text) - 1))
        ent["end_idx"] = max(0, min(ent["end_idx"], len(window_text) - 1))

    return window_text, subj_w, obj_w

## Tokenize Examples and Preserve Sample Weights

The tokenization step inserts entity markers around the subject and object mentions and builds a local text window around the entity pair.

For **Run 1**, tokenized examples also preserve a `sample_weight` field.
This weight is not used by the tokenizer itself, but it will later be passed to the custom trainer so that higher-quality annotations contribute more strongly to the loss.

In [ ]:
def tokenize_re_example(
    example,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    fallback_to_fulltext=True,
):
    w_text, w_subj, w_obj = build_window_around_entities(
        example["text"], example["subject"], example["object"], window_chars=window_chars
    )
    marked_text = insert_entity_markers(w_text, w_subj, w_obj)

    encoding = tokenizer(
        marked_text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].squeeze(0)
    attention_mask = encoding["attention_mask"].squeeze(0)

    e1_mask = (input_ids == e1_token_id).long()
    e2_mask = (input_ids == e2_token_id).long()

    if (e1_mask.sum().item() != 1 or e2_mask.sum().item() != 1) and fallback_to_fulltext:
        marked_text = insert_entity_markers(example["text"], example["subject"], example["object"])
        if marked_text.count("[E1]") != 1 or marked_text.count("[E2]") != 1:
            return None

        encoding = tokenizer(
            marked_text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        e1_mask = (input_ids == e1_token_id).long()
        e2_mask = (input_ids == e2_token_id).long()

    if e1_mask.sum().item() != 1 or e2_mask.sum().item() != 1:
        return None

    label = label2id[example["predicate"]]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "e1_mask": e1_mask,
        "e2_mask": e2_mask,
        "labels": torch.tensor(label, dtype=torch.long),
        "sample_weight": torch.tensor(example.get("sample_weight", 1.0), dtype=torch.float),
    }


print("Tokenization functions defined")

## Build Cached Tokenized Datasets with Weights

To speed up repeated experiments, the examples are pre-tokenized once and stored on disk.

For **Run 1**, the cached items include:
- token ids
- attention mask
- entity marker masks
- class label
- sample weight

The collator dynamically pads the batch and preserves all fields needed by the custom trainer.

In [ ]:
from dataclasses import dataclass
from transformers import PreTrainedTokenizerBase

@dataclass
class REDataCollatorWithPadding:
    tokenizer: PreTrainedTokenizerBase
    pad_to_multiple_of: int | None = None

    def __call__(self, features):
        labels = torch.stack([f["labels"] for f in features])
        sample_weight = torch.stack([f["sample_weight"] for f in features])

        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]

        def pad_1d(x, pad_value=0):
            if x.shape[0] == max_len:
                return x
            out = torch.full((max_len,), pad_value, dtype=x.dtype)
            out[:x.shape[0]] = x
            return out

        e1 = torch.stack([pad_1d(f["e1_mask"]) for f in features])
        e2 = torch.stack([pad_1d(f["e2_mask"]) for f in features])

        batch["e1_mask"] = e1
        batch["e2_mask"] = e2
        batch["labels"] = labels
        batch["sample_weight"] = sample_weight
        return batch

In [13]:
# CACHE_DIR = os.path.join(output_model_dir, "cache_tok")
CACHE_DIR = "../models/bert_biomedbert_re_A2_weighted_fixed/cache_tok"
os.makedirs(CACHE_DIR, exist_ok=True)

TRAIN_CACHE = os.path.join(CACHE_DIR, f"train_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")
DEV_CACHE   = os.path.join(CACHE_DIR, f"dev_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")

class ListREDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def pretokenize_examples_dynamic(
    examples,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    cache_path=None,
    verbose_every=5000,
):
    if cache_path is not None and os.path.exists(cache_path):
        print(f"[cache] Loading dynamic tokenized dataset from: {cache_path}")
        payload = torch.load(cache_path, map_location="cpu")
        return payload["items"], payload.get("skipped", [])

    print("[cache] Building dynamic tokenized items... (runs once)")
    items = []
    skipped = []

    for i, ex in enumerate(tqdm(examples, desc="Pre-tokenizing(dyn)", total=len(examples))):
        out = None
        try:
            out = tokenize_re_example(
                ex,
                tokenizer,
                e1_token_id,
                e2_token_id,
                max_length=max_length,
                window_chars=window_chars,
                fallback_to_fulltext=True,
            )
        except Exception as e:
            skipped.append((ex.get("pmid"), f"exception:{type(e).__name__}:{str(e)[:120]}"))
            continue

        if out is None:
            skipped.append((ex.get("pmid"), "tokenize_returned_None"))
            continue

        if out["e1_mask"].sum().item() != 1 or out["e2_mask"].sum().item() != 1:
            skipped.append((ex.get("pmid"), f"bad_markers_e1={out['e1_mask'].sum().item()}_e2={out['e2_mask'].sum().item()}"))
            continue

        items.append({
            "input_ids": out["input_ids"].to(torch.int64),
            "attention_mask": out["attention_mask"].to(torch.int64),
            "e1_mask": out["e1_mask"].to(torch.int64),
            "e2_mask": out["e2_mask"].to(torch.int64),
            "labels": out["labels"].to(torch.int64),
            "sample_weight": out["sample_weight"].to(torch.float32),
        })

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"  ...processed {i+1}/{len(examples)} | kept={len(items)} | skipped={len(skipped)}")

    print(f"[cache] Done. kept={len(items)} / {len(examples)} | skipped={len(skipped)}")

    if cache_path is not None:
        torch.save({"items": items, "skipped": skipped}, cache_path)
        print(f"[cache] Saved dynamic tokenized dataset to: {cache_path}")

        skip_txt = cache_path.replace(".pt", "_skipped.txt")
        with open(skip_txt, "w", encoding="utf-8") as f:
            for pmid, reason in skipped:
                f.write(f"{pmid}\t{reason}\n")
        print(f"[cache] Saved skipped list to: {skip_txt}")

    return items, skipped


train_items, train_skipped = pretokenize_examples_dynamic(
    train_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=TRAIN_CACHE
)

dev_items, dev_skipped = pretokenize_examples_dynamic(
    dev_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=DEV_CACHE
)

train_dataset = ListREDataset(train_items)
dev_dataset = ListREDataset(dev_items)

collator = REDataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print("FAST datasets ready!")
print(f"Train kept: {len(train_dataset)} | skipped: {len(train_skipped)}")
print(f"Dev kept: {len(dev_dataset)} | skipped: {len(dev_skipped)}")

Pre-tokenizing(dyn):  67%|██████▋   | 215486/319650 [01:19<00:37, 2745.00it/s]

  ...processed 215000/319650 | kept=214980 | skipped=20


Pre-tokenizing(dyn):  69%|██████▉   | 220412/319650 [01:21<00:35, 2789.19it/s]

  ...processed 220000/319650 | kept=219980 | skipped=20


Pre-tokenizing(dyn):  70%|███████   | 225314/319650 [01:23<00:34, 2767.41it/s]

...

  ...processed 315000/319650 | kept=314968 | skipped=32


Pre-tokenizing(dyn): 100%|██████████| 319650/319650 [02:00<00:00, 2655.47it/s]


[cache] Done. kept=319618 / 319650 | skipped=32
[cache] Saved dynamic tokenized dataset to: ../models/bert_biomedbert_re_A2_weighted_fixed/cache_tok\train_dyn_maxlen512_win300.pt
[cache] Saved skipped list to: ../models/bert_biomedbert_re_A2_weighted_fixed/cache_tok\train_dyn_maxlen512_win300_skipped.txt
[cache] Building dynamic tokenized items... (runs once)


Pre-tokenizing(dyn): 100%|██████████| 2229/2229 [00:00<00:00, 2697.84it/s]


[cache] Done. kept=2229 / 2229 | skipped=0
[cache] Saved dynamic tokenized dataset to: ../models/bert_biomedbert_re_A2_weighted_fixed/cache_tok\dev_dyn_maxlen512_win300.pt
[cache] Saved skipped list to: ../models/bert_biomedbert_re_A2_weighted_fixed/cache_tok\dev_dyn_maxlen512_win300_skipped.txt
FAST datasets ready!
Train kept: 319618 | skipped: 32
Dev kept: 2229 | skipped: 0


## Initialize the Entity-Marker BERT Model

The model architecture remains the same as in the baseline:
- a pretrained biomedical BERT encoder
- special entity markers for subject and object mentions
- concatenation of the marker representations
- a classification head over relation labels

For **Run 1**, the main change is not the encoder architecture itself, but the **training strategy**:
weighted supervision and a better validation metric.

In [14]:
class BertForREWithEntityMarkers(nn.Module):
    """
    BERT model for Relation Extraction with entity marker tokens.
    """

    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)

        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Linear(hidden_size * 2, num_labels)

        self.num_labels = num_labels

    def forward(
        self,
        input_ids,
        attention_mask,
        e1_mask,
        e2_mask,
        labels=None,
        sample_weight=None,
        **kwargs
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state

        e1_h = torch.bmm(e1_mask.unsqueeze(1).float(), sequence_output).squeeze(1)
        e2_h = torch.bmm(e2_mask.unsqueeze(1).float(), sequence_output).squeeze(1)

        concat_h = torch.cat([e1_h, e2_h], dim=-1)
        concat_h = self.dropout(concat_h)

        logits = self.classifier(concat_h)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(reduction="none")
            per_example_loss = loss_fct(logits, labels)

            if sample_weight is not None:
                loss = (per_example_loss * sample_weight).mean()
            else:
                loss = per_example_loss.mean()

        return {
            "loss": loss,
            "logits": logits
        }


print("BERT RE model class defined")

print("Initializing BERT RE model...")
model = BertForREWithEntityMarkers(model_name, num_labels=len(RELATION_LABELS))
model.bert.resize_token_embeddings(len(tokenizer))

print(f"Model initialized")
print(f"  Number of labels: {model.num_labels}")
print(f"  Hidden size: {model.bert.config.hidden_size}")

BERT RE model class defined
Initializing BERT RE model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1167.08it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not

Model initialized
  Number of labels: 18
  Hidden size: 768


## Define a Custom Trainer with Weighted Loss and Positive-Class F1

This trainer introduces two important improvements over the baseline:

### 1. Weighted loss
Each example contributes to the loss according to its annotation quality:
- cleaner annotations have higher influence
- noisier annotations still contribute, but less strongly

### 2. Better model selection metric
Instead of selecting checkpoints using validation loss, **Run 1** selects the best checkpoint using a positive-relation F1 score.

This is more appropriate for relation extraction because the dataset contains many `"no relation"` examples, and loss alone may not reflect true extraction quality.

In [15]:
import os
import torch
import numpy as np
from sklearn.metrics import f1_score
from transformers import Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    pos_label_ids = [idx for label, idx in label2id.items() if label != "no relation"]

    macro_f1_pos = f1_score(
        labels,
        preds,
        labels=pos_label_ids,
        average="macro",
        zero_division=0
    )

    micro_f1_pos = f1_score(
        labels,
        preds,
        labels=pos_label_ids,
        average="micro",
        zero_division=0
    )

    return {
        "macro_f1_pos": macro_f1_pos,
        "micro_f1_pos": micro_f1_pos,
    }


class RETrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

        sample_weight = inputs.pop("sample_weight", None)

        outputs = model(**inputs, sample_weight=sample_weight)

        loss = outputs["loss"]

        return (loss, outputs) if return_outputs else loss

    def _save(self, output_dir: str, state_dict=None):
        os.makedirs(output_dir, exist_ok=True)

        if state_dict is None:
            state_dict = self.model.state_dict()

        for k, v in state_dict.items():
            if isinstance(v, torch.Tensor) and not v.is_contiguous():
                state_dict[k] = v.contiguous()

        torch.save(state_dict, os.path.join(output_dir, "pytorch_model.bin"))
        torch.save(self.args, os.path.join(output_dir, "training_args.bin"))

## Configure Training Arguments for Run 1

The training configuration for **Run 1** is designed to be a solid first experiment:
- standard learning rate for biomedical BERT fine-tuning
- moderate batch size
- linear schedule with warmup
- model selection based on **validation macro F1 over positive relation labels**

This is more appropriate than selecting the best checkpoint using validation loss, because relation extraction performance is better reflected by label-level F1 than by overall loss.

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    learning_rate=2e-5,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    max_grad_norm=1.0,

    num_train_epochs=3,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1_pos",
    greater_is_better=True,

    disable_tqdm=False,
    logging_steps=50,

    fp16=torch.cuda.is_available(),
    tf32=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none"
)

print("Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Best model metric: {training_args.metric_for_best_model}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training configuration ready
  Batch size: 16
  Epochs: 3
  Learning rate: 2e-05
  Best model metric: eval_macro_f1_pos


## Initialize the Trainer

The trainer combines:
- the weighted entity-marker BERT model
- the cached training and development datasets
- the custom collator
- the positive-class F1 evaluation metric

This setup implements the full **Run 1** pipeline.

In [17]:
trainer = RETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

Trainer initialized
  Training samples: 319618
  Evaluation samples: 2229


## Train the Model

This cell starts the fine-tuning process for **Run 1**.

Main characteristics of this run:
- mixed-quality supervision
- source-aware sample weighting
- entity-marker representation
- dynamic padding
- best checkpoint selected using positive-relation macro F1

This provides a stronger and more realistic first run than a baseline that treats all annotations equally and selects checkpoints using validation loss.

In [18]:
print(train_dataset[0].keys())
print(train_dataset[0])

dict_keys(['input_ids', 'attention_mask', 'e1_mask', 'e2_mask', 'labels', 'sample_weight'])
{'input_ids': tensor([    2, 30072,    16,    43,  9316,  4966,    13,    18, 15390,  7514,
         2518,  2868,  1956, 19793,  2014, 19713,  3201,  1024,     7,    23,
         2430,  2567,  2502, 30029, 11483,  2367,    16,  2502,  6139,  8471,
         1930,  3261,  8828,    16,  2452,  1942,  1920,  6829,  2210,    18,
        19793,  2014, 21484,  2311,  3261,  1920,  2294,  1927, 12898,    16,
        15390,  7514,    16,  1930,  2502, 30522,   160,    17, 11953, 30523,
           16, 11906,    17,    21,    16,  1930, 19609,    17,    25,  1922,
         1920, 30524,  4120, 30525,    16,  6071,  6670,    16,  1930,  3068,
           18,  2222,    16, 19793,  2014, 21484,  2311,  4256,  1920,  2308,
        28822,    17, 22051,    43, 10027, 13763,    16,    32,    51,    34,
         2308,  3553,  2068,  3693,  3940,  7953, 23854, 14937,  2099,    32,
           19,    51,    34,    16, 

In [ ]:
print("=" * 60)
print("Starting model training...")
print("=" * 60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "=" * 60)
print("TRAINING COMPLETED!")
print("=" * 60)
print(f"Training time: {training_duration / 60:.2f} minutes")

eval_metrics = trainer.evaluate()
print("\nFinal evaluation metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v}")

Starting model training...


Step,Training Loss,Validation Loss


## Save the Trained Model

After training, the model, tokenizer, and label mappings are saved to disk.

The output directory contains everything needed to reload the fine-tuned relation extraction model for inference or further experiments.

In [ ]:
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

with open(os.path.join(output_model_dir, "label_mappings.json"), "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

print(f"Model saved to: {output_model_dir}")